In [1]:
import os

In [2]:
%pwd

'd:\\Data Science\\project Series\\End_to_end_ReD_Wine_Quality_Mlops_Project\\research'

In [3]:
os.chdir('../')

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_file_path: Path


In [5]:
from WineQuality_Project.constants import *
from WineQuality_Project.utils.common import read_yaml,create_directories

In [7]:

class ConfigManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([Path(self.config['artifact_root'])])


    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config['model_evaluation']
        Path(config['root_dir']).mkdir(parents=True, exist_ok=True)
        
        return ModelEvaluationConfig(
            root_dir=Path(config['root_dir']),
            test_data_path=Path(config['test_data_path']),
            model_path=Path(config['model_path']),
            metric_file_path=Path(config['metric_file_name'])
        )


In [8]:
import pandas as pd
import joblib
import json
import logging
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig, target_column: str):
        self.config = config
        self.target_column = target_column

    def evaluate(self):
        try:
            # Load test data
            df_test = pd.read_csv(self.config.test_data_path)
            logging.info(f"Test data loaded from: {self.config.test_data_path}")

            X_test = df_test.drop([self.target_column], axis=1)
            y_test = df_test[self.target_column]

            # Load trained model
            model = joblib.load(self.config.model_path)
            logging.info(f"Model loaded from: {self.config.model_path}")

            # Make predictions
            y_pred = model.predict(X_test)

            # Calculate metrics
            metrics = {
                "r2_score": r2_score(y_test, y_pred),
                "mse": mean_squared_error(y_test, y_pred),
                "mae": mean_absolute_error(y_test, y_pred)
            }

            # Save metrics to JSON
            self.config.metric_file_path.parent.mkdir(parents=True, exist_ok=True)
            with open(self.config.metric_file_path, 'w') as f:
                json.dump(metrics, f, indent=4)

            logging.info(f"Metrics saved at: {self.config.metric_file_path}")
            logging.info(f"Model evaluation metrics: {metrics}")

            return metrics

        except Exception as e:
            logging.error(f"Model evaluation failed: {e}")
            raise e


In [11]:

STAGE_NAME = "Model Evaluation Stage"
from WineQuality_Project import logger
try:
    logger.info(f">>> Stage {STAGE_NAME} started")

    # 1️⃣ Initialize config manager
    config_manager = ConfigManager()

    # 2️⃣ Get model evaluation config
    eval_config = config_manager.get_model_evaluation_config()

    # 3️⃣ Initialize ModelEvaluation
    evaluator = ModelEvaluation(config=eval_config, target_column="quality")

    # 4️⃣ Run evaluation
    metrics = evaluator.evaluate()

    logger.info(f">>> Stage {STAGE_NAME} completed successfully")
    print("Evaluation Metrics:", metrics)

except Exception as e:
    logger.exception(f"{STAGE_NAME} failed: {e}")
    raise e

[2025-12-12 21:14:15] [INFO] WineQualityLogger - >>> Stage Model Evaluation Stage started
[2025-12-12 21:14:15] [INFO] WineQualityLogger - YAML file: config\config.yml loaded successfully
[2025-12-12 21:14:15] [INFO] WineQualityLogger - YAML file: params.yaml loaded successfully
[2025-12-12 21:14:15] [INFO] WineQualityLogger - YAML file: schema.yaml loaded successfully
[2025-12-12 21:14:15] [INFO] WineQualityLogger - Directory created at: artifacts
[2025-12-12 21:14:15] [INFO] WineQualityLogger - >>> Stage Model Evaluation Stage completed successfully
INFO:WineQualityLogger:>>> Stage Model Evaluation Stage completed successfully


Evaluation Metrics: {'r2_score': 0.45580350193595154, 'mse': 0.30283065171345075, 'mae': 0.42301696770438635}
